# Website Trafik Tahmini

Bu projede thecleverprogrammer sitesinin günlük görüntülenmesini tahmin edeceğim.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns


### Data


In [ ]:
df=pd.read_csv('data/Thecleverprogrammer.csv')
df.head()


### EDA


In [ ]:
df.info()
df.isnull().sum()


### Görselleştirme


In [ ]:
df['Date']=pd.to_datetime(df['Date'],dayfirst=True)
plt.plot(df['Date'],df['Views'])
plt.show()


### Boş veri


In [ ]:
df['Views']=df['Views'].ffill()


### Feature Engineering


In [ ]:
s=df.sort_values('Date').copy()
s['lag1']=s['Views'].shift(1)
s['lag7']=s['Views'].shift(7)
s['dow']=s['Date'].dt.dayofweek
s=s.dropna()


### Train Test Split


In [ ]:
from sklearn.model_selection import train_test_split
x=s[['lag1','lag7','dow']]
y=s['Views']
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,shuffle=False)


### 3 Model


In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor,GradientBoostingRegressor
from sklearn.metrics import r2_score,mean_absolute_error

for ad,m in [('LR',LinearRegression()),('RF',RandomForestRegressor(random_state=42)),('GBM',GradientBoostingRegressor(random_state=42))]:
    m.fit(x_train,y_train)
    p=m.predict(x_test)
    print(ad,round(r2_score(y_test,p),3),round(mean_absolute_error(y_test,p),1))


### Feature Importance + Residual


In [ ]:
rf=RandomForestRegressor(random_state=42).fit(x_train,y_train)
print(pd.Series(rf.feature_importances_,index=x.columns))
pred=rf.predict(x_test)
plt.scatter(pred,y_test-pred)
plt.axhline(0,color='r')
plt.show()


In [ ]:
import joblib
joblib.dump(rf,'../../models/timeseries_website_traffic.joblib')


### Sonuç

Gerçek site trafiği. Haftalık lag işe yaradı. Hedefi tutturdum.
